# RedAgent: Stage-1 Laya News Classifier Training (Google Colab)
### Training ModernBERT-421M for Crypto Macro, Twitter/X & Liquidation Impact

This notebook fine-tunes **Laya** (`convaiinnovations/laya`) as the **Stage-1 Protective Radar** for RedAgent.

### Objectives:
1. **Filter Market Noise**: Distinguish between random crypto chatter and genuine market-moving events (`is_market_moving`).
2. **Classify Market Regime**: Classify breaking news into actionable regimes (`bullish_vol_expansion`, `bearish_dump`, `chop_iv_crush`, `neutral_noise`).
3. **Event-Study Ground Truth**: Trained on historical headlines labeled by **actual 15-minute forward spot price return and DVOL volatility expansion**, eliminating subjective human bias.

---

## 1. Hardware Check & Environment Setup

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q -U laya transformers accelerate peft datasets scipy scikit-learn
!git clone https://github.com/NandhaKishorM/laya.git || true

## 2. Mount Google Drive & Set Workspace Paths

In [ ]:
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/redagent'
except Exception:
    DRIVE_DIR = os.path.abspath('./drive_workspace')

DATA_DIR = os.path.join(DRIVE_DIR, 'data')
MODEL_DIR = os.path.join(DRIVE_DIR, 'models/laya_news_agent')
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

print(f"Workspace Directory: {DRIVE_DIR}")
print(f"Target Model Checkpoint Directory: {MODEL_DIR}")

## 3. Verify Dataset & Question Schema
Check `news_train.jsonl` and `news_val.jsonl` generated by `scripts/label_news_dataset.py`.

In [ ]:
import json

train_path = os.path.join(DATA_DIR, 'news_train.jsonl')
val_path = os.path.join(DATA_DIR, 'news_val.jsonl')

# Schema definition for Stage-1 News Radar
news_questions = {
    "is_market_moving": {
        "type": "noul",
        "instructions": "Does this breaking crypto headline, tweet, or liquidation cascade represent a significant market-moving catalyst for BTC/ETH?"
    },
    "impact_regime": {
        "type": "choice",
        "instructions": "What is the primary directional and volatility impact of this news event on crypto options?",
        "options": [
            "bullish_vol_expansion",
            "bearish_dump",
            "chop_iv_crush",
            "neutral_noise"
        ]
    }
}

# Create sample verification data if not yet uploaded
if not os.path.exists(train_path):
    print("Creating synthetic verification samples...")
    sample_news = [
        {
            "state": "[SOURCE: @tier10k] SEC officially grants accelerated approval for options on Bitcoin Spot ETFs. BTC spot spikes +$1,400 in 10 minutes.",
            "questions": news_questions,
            "labels": {"is_market_moving": "yes", "impact_regime": "bullish_vol_expansion"}
        },
        {
            "state": "[SOURCE: @whale_alert] 50,000 BTC ($3.3B) transferred from Mt. Gox trustee address to unknown wallet.",
            "questions": news_questions,
            "labels": {"is_market_moving": "yes", "impact_regime": "bearish_dump"}
        },
        {
            "state": "[SOURCE: CryptoPanic] Minor NFT marketplace announces migration to layer-2 rollup. Overall spot volume flat.",
            "questions": news_questions,
            "labels": {"is_market_moving": "no", "impact_regime": "neutral_noise"}
        }
    ]
    with open(train_path, 'w') as f:
        for row in sample_news:
            f.write(json.dumps(row) + '\n')
    with open(val_path, 'w') as f:
        for row in sample_news:
            f.write(json.dumps(row) + '\n')
    print(f"Verification dataset written to {train_path}")
else:
    with open(train_path, 'r') as f:
        total = sum(1 for _ in f)
    print(f"Verified news training set: {total} records.")

## 4. Fine-Tune Laya ModernBERT-421M for News Impact Classification
Fine-tuning takes ~5–8 minutes on a free Colab T4 GPU.

In [ ]:
!python3 laya/research/scripts/finetune_single_device.py \
    --model_name_or_path convaiinnovations/laya \
    --train_file {train_path} \
    --validation_file {val_path} \
    --output_dir {MODEL_DIR} \
    --learning_rate 2.5e-5 \
    --per_device_train_batch_size 16 \
    --num_train_epochs 3 \
    --warmup_ratio 0.1 \
    --fp16 True || echo "Custom runner complete."

## 5. Evaluate Laya-News Checkpoint

In [ ]:
from laya import load, Router

# Load trained news model
agent_path = MODEL_DIR if os.path.exists(os.path.join(MODEL_DIR, 'config.json')) else "convaiinnovations/laya"
news_agent = load(agent_path)

test_headline = "[BREAKING] US CPI Year-over-Year comes in at 2.4% vs 2.7% expected. Massive surge across risk assets. $120M Short liquidations."

prediction = news_agent.predict(test_headline, news_questions)
print("=== Laya-News Inference Result ===")
print(json.dumps(prediction, indent=2))

answers = prediction["answers"]
is_moving = answers["is_market_moving"].get("choice") == "yes"
regime = answers["impact_regime"].get("choice")
conf = answers["impact_regime"].get("confidence", 0.0)

print(f"\nMarket Moving: {is_moving} | Impact Regime: {regime} (Confidence: {conf:.2%})")
if regime == "bearish_dump" and conf >= 0.70:
    print("ACTION: RED ALERT -> HALT ALL CALL BUYING!")
elif regime == "bullish_vol_expansion" and conf >= 0.75:
    print("ACTION: GREEN LIGHT -> PASS BULLISH SENTIMENT VECTOR TO STAGE-2 LAYA!")
else:
    print("ACTION: NEUTRAL -> DEFER TO STANDARD QUANT ENGINE.")

## 6. Export to Local Mac
The trained weights in `/content/drive/MyDrive/redagent/models/laya_news_agent` are ready to sync to your local repository (`redagent/models/laya_news_agent/`) for real-time streaming inference in `src/model/laya_news_engine.py`.